# Homework: Agentic RAG

A RAG system built from scratch and then made **agentic** — the same path
as the module. The knowledge base is the **course lessons themselves** (markdown
pages pulled from the course repo).

## Setup

Load the Anthropic API key an create the LLM client. Built with **Anthropic `claude-sonnet-4-5`** instead of OpenAI `gpt-5.4-mini`, so
the exact token counts in the exercises can differ a little.

In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()                 # reads ANTHROPIC_API_KEY from .env
client = Anthropic()

MODEL = "claude-sonnet-4-5"

## Fetch the lesson pages

Lesson markdown pulled straight from the course repo, pinned to commit
`8c1834d`.

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
# Each file has a parse() method returning {"filename": ..., "content": ...}
documents = [file.parse() for file in files]

In [4]:
# Peek at one document
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q1. How many lesson pages

Each lesson page is one markdown file, so the number of documents is the number
of lesson pages.

In [5]:
print("Q1 — lesson pages:", len(documents))

Q1 — lesson pages: 72


**Answer Q1: 72**

## Q2. Indexing and searching

Index the documents with `minsearch`: `content` as a text field and `filename`
as a keyword field. Then run the test query and look at the top result.

In [6]:
from minsearch import Index

# Index over whole documents
doc_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
doc_index.fit(documents)

In [7]:
question = "How does the agentic loop keep calling the model until it stops?"

results = doc_index.search(question, num_results=5)
print("Q2 — top result:", results[0]["filename"])

Q2 — top result: 01-agentic-rag/lessons/14-agentic-loop.md


**Answer Q2: `01-agentic-rag/lessons/14-agentic-loop.md`**

## Q3. RAG

A RAG assistant built on the index with `rag_helper.py` from the lessons,
adapted for the `filename`/`content` schema:

- `search` queries the `minsearch` index. For now, `boost_dict` and `filter_dict` are not used.
- `build_context` formats `filename` + `content`,
- `llm` returns the **whole response** (not just text) and `rag` returns it too,
  so `response.usage` can be read to count the tokens.

In [8]:
from rag_helper import RAGBase

rag_assistant = RAGBase(
    index=doc_index,
    llm_client=client,
    model=MODEL,
)

In [9]:
response = rag_assistant.rag(question)

print("Answer:\n" + response.content[0].text)
print("\nQ3 — input tokens:", response.usage.input_tokens)

Answer:
# Answer

The agentic loop keeps calling the model until it stops by using a **`has_function_calls` flag** that tracks whether the model's response contains any function calls.

Here's how it works:

## The Core Mechanism

1. **Initialize the flag**: At the start of each iteration, `has_function_calls` is set to `False`

2. **Check the response**: After getting a response from the model, the code iterates through all output items:
   - If it finds a `function_call` type, it sets `has_function_calls = True`
   - If it finds a `message` type, it just displays the message

3. **Exit condition**: At the end of each iteration, the loop checks:
   ```python
   if has_function_calls == False:
       break
   ```

## The Loop Structure

```python
while True:
    has_function_calls = False
    
    # Call the model
    response = openai_client.responses.create(...)
    
    # Process the response
    for item in response.output:
        if item.type == "function_call":
            has_f

**Answer Q3: ~7000 input tokens** (closest option).

## Q4. Chunking

As asked, chunking is performed with the `chunk_documents` method from `gitsource`

In [10]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print("Q4 — chunks:", len(chunks))

Q4 — chunks: 295


**Answer Q4: 295**

## Q5. RAG with chunking

Index the **chunks** the same way, point a RAG at the chunk index, and answer the
same query. Each chunk is smaller, so the context shrinks — the input-token count
drops compared to Q3.

In [11]:
# Defined new index and assistant for chunks
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
chunk_index.fit(chunks)

chunk_rag_assistant = RAGBase(
    index=chunk_index,
    llm_client=client,
    model=MODEL,
)

In [12]:
chunk_response = chunk_rag_assistant.rag(question)

q3_tokens = response.usage.input_tokens
q5_tokens = chunk_response.usage.input_tokens

print("Q3 input tokens:", q3_tokens)
print("Q5 input tokens:", q5_tokens)
print("(Q3 / Q5): {:.1f}x fewer".format(q3_tokens / q5_tokens))

Q3 input tokens: 8213
Q5 input tokens: 2669
(Q3 / Q5): 3.1x fewer


**Answer Q5: ~3× fewer input tokens.** 

## Q6. Turning it into an agent

So far search runs once with the exact query. The agentic version gives the LLM a
`search` tool and lets it decide when (and what) to search. Built with
[`toyaikit`](https://github.com/alexeygrigorev/toyaikit) and its
`AnthropicMessagesRunner`, the same way as in the ToyAIKit lesson.

In [13]:
def search(query: str) -> list:
    """
    Search the course materials for content relevant to the query.

    Use this to look up information from the course lessons before
    answering a student's question. Call it multiple times with
    different keywords to gather more context.

    Args:
        query: The search query, e.g. a few keywords or a question.

    Returns:
        A list of the most relevant chunks from the course lessons.
    """
    return chunk_index.search(query, num_results=5)

In [14]:
from toyaikit.tools import Tools
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import AnthropicMessagesRunner
from toyaikit.llm import AnthropicClient

agent_instructions = (
    "You're a course teaching assistant. Answer the student's question "
    "using the search tool. Make multiple searches with different keywords "
    "before answering."
)

# Register the search function as a tool (schema built from hints + docstring).
tools = Tools()
tools.add_tool(search)

runner = AnthropicMessagesRunner(
    tools=tools,
    developer_prompt=agent_instructions,
    chat_interface=IPythonChatInterface(),
    llm_client=AnthropicClient(model=MODEL),
)

In [15]:
# The agent decides on its own when to search and when to answer
agent_question = "How does the agentic loop work, and how is it different from plain RAG?"

result = runner.loop(prompt=agent_question, callback=runner.displaying_callback)

-> Response received


-> Response received


-> Response received


-> Response received


Aspect,Plain RAG,Agentic Loop
Control,Fixed pipeline,LLM decides
Search frequency,Exactly once,Multiple times if needed
Error recovery,None,Can retry with corrections
Adaptability,Always same steps,Adapts based on results
Cost,Single LLM call,Multiple calls (higher cost)
Complexity,Simple,More complex


In [16]:
# Each tool result is a message with role "tool" — count them.
search_calls = sum(
    1 for m in result.new_messages
    if isinstance(m, dict) and m.get("role") == "tool"
)
print("Q6 — times the agent called search:", search_calls)

Q6 — times the agent called search: 8


**Answer Q6: ~10 search calls** (closest option). The agent decides this itself, so it varies between runs — this run made **8** calls.